# Dataset 作成
- 前回の面接に合わせたデータセットを作成
    - 商品マスタ
    - 各店舗の売り上げデータ
    - 各店舗の在庫データ

In [0]:
#  ライブラリインポート
import random
from datetime import date, timedelta
from databricks.connect import DatabricksSession

In [0]:
random.seed(0)

# 商品マスタ
categories = ["飲料", "食品", "日用品", "菓子", "酒類"]
product_names = [
    "ミネラルウォーター 500ml", "緑茶 500ml", "コーラ 500ml", "コーヒー 缶",
    "カップ麺 醤油味", "おにぎり 鮭", "サンドイッチ ハム", "冷凍餃子",
    "ティッシュペーパー 5箱", "洗剤 詰め替え", "歯ブラシ",
    "ポテトチップス", "チョコレート", "ガム",
    "缶ビール 350ml", "ハイボール 350ml"
]

products = [
    {
        "product_id": f"P{idx + 1:03}",
        "product_name": name,
        "category": random.choice(categories),
        "unit_price": random.choice([98, 128, 158, 198, 258, 298, 398, 498]),
    }
    for idx, name in enumerate(product_names)
]

df_products = spark.createDataFrame(products)
df_products.write.mode("overwrite").saveAsTable("workspace.practice.products")

display(df_products)

print(f"商品マスタ作成完了: {len(products)}件")

In [0]:
# 各店舗の売り上げデータ
stores = [
    {"store_id": "S01", "store_name": "渋谷店"},
    {"store_id": "S02", "store_name": "新宿店"},
    {"store_id": "S03", "store_name": "池袋店"},
    {"store_id": "S04", "store_name": "横浜店"},
]

start_date = date(2026, 6, 1)
sales = []
sale_id = 1
for i in range(30):
    sale_date = start_date + timedelta(days=i)
    for store in stores:
        # 1日あたり店舗ごとに数件のランダムな売り上げを生成
        for _ in range(random.randint(3, 8)):
            product = random.choice(products)
            quantity = random.randint(1, 10)
            sales.append({
                "sale_id": f"SL{sale_id:05}",
                "store_id": store["store_id"],
                "store_name": store["store_name"],
                "product_id": product["product_id"],
                "sale_date": sale_date.isoformat(),
                "quantity": quantity,
                "sales_amount": quantity * product["unit_price"],
            })
            sale_id += 1

df_sales = spark.createDataFrame(sales)
df_sales.write.mode("overwrite").saveAsTable("workspace.practice.sales")

print(f"売り上げデータ作成完了: {len(sales)}件")
display(df_sales)

In [0]:
# 各店舗の在庫データ
inventory = [
    {
        "store_id": store["store_id"],
        "store_name": store["store_name"],
        "product_id": product["product_id"],
        "stock_quantity": random.randint(0, 100),
        "updated_at": start_date.isoformat(),
    }
    for store in stores
    for product in products
]

df_inventory = spark.createDataFrame(inventory)
df_inventory.write.mode("overwrite").saveAsTable("workspace.practice.inventory")

print(f"在庫データ作成完了: {len(inventory)}件")
display(df_inventory)


In [0]:
# 作成したテーブルの確認
spark.read.table("workspace.practice.products").show(5)
spark.read.table("workspace.practice.sales").show(5)
spark.read.table("workspace.practice.inventory").show(5)